In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ALQ_E.xpt"
csv_file = DATA_DIR / "ALQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 ALQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ALQ101",
    "ALQ110",
    "ALQ120Q",
    "ALQ120U",
    "ALQ130",
    "ALQ140Q",
    "ALQ140U",
    "ALQ150"
]


if df.shape != (5707, 9):

    raise ValueError(
        f"Unexpected ALQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "ALQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():

    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: ALQ_E structure = "
    "5,707 rows x 9 columns."
)


# ============================================================
# 4. KNOWN SAS/XPORT TINY-ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(true_zero_counts.sum()):,}"
)

print(
    "Tiny artifacts:",
    f"{int(tiny_counts.sum()):,}"
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 5. EXPECTED TINY PATTERN
# ============================================================

EXPECTED_TINY_COUNTS = {
    "ALQ120Q": 1111,
    "ALQ140Q": 2036
}

EXPECTED_TINY_TOTAL = 3147


actual_tiny_counts = {
    col: int(count)
    for col, count in tiny_counts.items()
    if count > 0
}


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny artifact pattern differs.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if int(
    tiny_counts.sum()
) != EXPECTED_TINY_TOTAL:

    raise ValueError(
        "Unexpected total tiny artifact count."
    )


if int(
    true_zero_counts.sum()
) != 0:

    raise ValueError(
        "Unexpected true numeric zero already "
        "decoded from source XPT."
    )


print(
    "\nPASS: Exactly 3,147 tiny artifacts "
    "found in ALQ120Q and ALQ140Q only."
)


# ============================================================
# 6. ZERO-SAFETY CHECK
# ============================================================

#
# CDC:
#
# ALQ120Q:
# ENTER '0' FOR NEVER
# Released range begins at 0.
#
# ALQ140Q:
# ENTER '0' FOR NONE
# Released range begins at 0.
#
# Therefore these are validated zero artifacts.
#

APPROVED_ZERO_COLUMNS = {
    "ALQ120Q",
    "ALQ140Q"
}


unexpected_tiny_columns = (
    set(actual_tiny_counts)
    -
    APPROVED_ZERO_COLUMNS
)


if unexpected_tiny_columns:

    raise ValueError(
        "STOP: Tiny values found in "
        "unexpected variables:\n"
        + str(
            sorted(unexpected_tiny_columns)
        )
    )


print(
    "PASS: Tiny values occur only in "
    "CDC zero-permitted variables."
)


# ============================================================
# 7. RESTORE VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = df[col].eq(TINY_VALUE)

    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} "
            f"tiny values, found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0.0


    print(
        f"{col:<10}: "
        f"{actual_count:,} tiny -> 0"
    )


print(
    "PASS: All 3,147 validated zeros restored."
)


# ============================================================
# 8. VERIFY ZERO COUNTS
# ============================================================

if int(
    df["ALQ120Q"]
    .eq(0)
    .sum()
) != 1111:

    raise ValueError(
        "ALQ120Q zero count differs."
    )


if int(
    df["ALQ140Q"]
    .eq(0)
    .sum()
) != 2036:

    raise ValueError(
        "ALQ140Q zero count differs."
    )


print(
    "PASS: ALQ120Q zeros = 1,111."
)

print(
    "PASS: ALQ140Q zeros = 2,036."
)


# ============================================================
# 9. VERIFY NO TINY ARTIFACT REMAINS
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny artifacts:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain."
    )


print(
    "PASS: No tiny artifacts remain."
)


# ============================================================
# 10. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN = 41475-51623, "
    "no missing, no duplicates."
)


# ============================================================
# 11. FRACTIONAL / DECIMAL CHECK
# ============================================================

#
# ALQ_E should contain NO genuine decimal variables.
#

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        fractional_columns[col] = (
            fractional_count
        )


print(
    "\nGenuine fractional variables:",
    fractional_columns
)


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal "
        "values found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 12. CDC FREQUENCY CHECK - ALQ101
# ============================================================

EXPECTED_ALQ101 = {
    1: 3670,
    2: 1557,
    9: 6
}


actual = (
    df["ALQ101"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_ALQ101:

    raise ValueError(
        "ALQ101 frequency mismatch.\n"
        f"Expected: {EXPECTED_ALQ101}\n"
        f"Actual:   {actual}"
    )


if int(
    df["ALQ101"]
    .isna()
    .sum()
) != 474:

    raise ValueError(
        "ALQ101 missing count differs."
    )


print(
    "\nPASS: ALQ101 matches CDC."
)


# ============================================================
# 13. CDC FREQUENCY CHECK - ALQ110
# ============================================================

EXPECTED_ALQ110 = {
    1: 765,
    2: 789,
    9: 9
}


actual = (
    df["ALQ110"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_ALQ110:

    raise ValueError(
        "ALQ110 frequency mismatch."
    )


if int(
    df["ALQ110"]
    .isna()
    .sum()
) != 4144:

    raise ValueError(
        "ALQ110 missing count differs."
    )


print(
    "PASS: ALQ110 matches CDC."
)


# ============================================================
# 14. CDC CHECK - ALQ120Q
# ============================================================

#
# CDC:
#
# 0-365 = 4,423
# 777   = 0
# 999   = 11
# Missing = 1,273
#
# Source zero count = 1,111
#

s = df["ALQ120Q"]


range_count = int(
    s.between(
        0,
        365
    )
    .sum()
)


if range_count != 4423:

    raise ValueError(
        f"ALQ120Q 0-365 count differs: "
        f"{range_count:,}"
    )


if int(
    s.eq(777)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected ALQ120Q=777."
    )


if int(
    s.eq(999)
    .sum()
) != 11:

    raise ValueError(
        "ALQ120Q=999 count differs."
    )


if int(
    s.isna()
    .sum()
) != 1273:

    raise ValueError(
        "ALQ120Q missing count differs."
    )


if int(
    s.eq(0)
    .sum()
) != 1111:

    raise ValueError(
        "ALQ120Q zero count differs."
    )


print(
    "\nPASS: ALQ120Q"
)

print(
    "      0-365 = 4,423"
)

print(
    "      0 = 1,111"
)

print(
    "      999 = 11"
)

print(
    "      Missing = 1,273"
)


# ============================================================
# 15. CDC CHECK - ALQ120U
# ============================================================

EXPECTED_ALQ120U = {
    1: 1383,
    2: 819,
    3: 1110
}


actual = (
    df["ALQ120U"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_ALQ120U:

    raise ValueError(
        "ALQ120U frequency mismatch."
    )


if int(
    df["ALQ120U"]
    .isna()
    .sum()
) != 2395:

    raise ValueError(
        "ALQ120U missing count differs."
    )


print(
    "PASS: ALQ120U matches CDC."
)


# ============================================================
# 16. CDC CHECK - ALQ130
# ============================================================

#
# CDC:
#
# 1-83 = 3,312
# 999  = 11
# Missing = 2,384
#

s = df["ALQ130"]


if int(
    s.between(
        1,
        83
    )
    .sum()
) != 3312:

    raise ValueError(
        "ALQ130 range count differs."
    )


if int(
    s.eq(999)
    .sum()
) != 11:

    raise ValueError(
        "ALQ130=999 count differs."
    )


if int(
    s.isna()
    .sum()
) != 2384:

    raise ValueError(
        "ALQ130 missing count differs."
    )


print(
    "PASS: ALQ130 matches CDC."
)


# ============================================================
# 17. CDC CHECK - ALQ140Q
# ============================================================

#
# CDC:
#
# 0-365 = 3,306
# 7777  = 0
# 9999  = 17
# Missing = 2,384
#
# Source zero count = 2,036
#

s = df["ALQ140Q"]


if int(
    s.between(
        0,
        365
    )
    .sum()
) != 3306:

    raise ValueError(
        "ALQ140Q range count differs."
    )


if int(
    s.eq(7777)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected ALQ140Q=7777."
    )


if int(
    s.eq(9999)
    .sum()
) != 17:

    raise ValueError(
        "ALQ140Q=9999 count differs."
    )


if int(
    s.isna()
    .sum()
) != 2384:

    raise ValueError(
        "ALQ140Q missing count differs."
    )


if int(
    s.eq(0)
    .sum()
) != 2036:

    raise ValueError(
        "ALQ140Q zero count differs."
    )


print(
    "\nPASS: ALQ140Q"
)

print(
    "      0-365 = 3,306"
)

print(
    "      0 = 2,036"
)

print(
    "      9999 = 17"
)

print(
    "      Missing = 2,384"
)


# ============================================================
# 18. CDC CHECK - ALQ140U
# ============================================================

EXPECTED_ALQ140U = {
    1: 348,
    2: 263,
    3: 659
}


actual = (
    df["ALQ140U"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_ALQ140U:

    raise ValueError(
        "ALQ140U frequency mismatch."
    )


if int(
    df["ALQ140U"]
    .isna()
    .sum()
) != 4437:

    raise ValueError(
        "ALQ140U missing count differs."
    )


print(
    "PASS: ALQ140U matches CDC."
)


# ============================================================
# 19. CDC CHECK - ALQ150
# ============================================================

EXPECTED_ALQ150 = {
    1: 765,
    2: 3665,
    9: 4
}


actual = (
    df["ALQ150"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_ALQ150:

    raise ValueError(
        "ALQ150 frequency mismatch."
    )


if int(
    df["ALQ150"]
    .isna()
    .sum()
) != 1273:

    raise ValueError(
        "ALQ150 missing count differs."
    )


print(
    "PASS: ALQ150 matches CDC."
)


# ============================================================
# 20. BUILD EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# expected_df =
#
# ORIGINAL XPT
#
# +
# ONLY:
#
# ALQ120Q tiny -> 0
# ALQ140Q tiny -> 0
#
# NOTHING ELSE.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 21. CLEAN WHOLE-NUMBER FORMATTING
# ============================================================

#
# We proved every released variable is a
# whole-number identifier/code/count.
#
# Examples:
#
# SEQN
# 41475.0 -> 41475
#
# ALQ120Q
# tiny -> 0
# 7.0 -> 7
#
# ALQ140Q
# tiny -> 0
#

for col in EXPECTED_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All ALQ_E fields converted "
    "to clean whole-number formatting."
)


# ============================================================
# 22. PRE-EXPORT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal expected XPT "
    "after ONLY validated zero restorations."
)


# ============================================================
# 23. EXPORT CSV
# ============================================================

#
# NO rounding.
# NO float_format.
# Missing -> blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 24. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 25. STRUCTURE CHECK AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT."
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure matches XPT."
)


# ============================================================
# 26. EXACT NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += differences


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 9 columns numerically match "
    "expected XPT values exactly."
)


# ============================================================
# 27. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 28. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 5707:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 29. STRICT RAW FORMAT CHECK
# ============================================================

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = row[col]


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(raw_examples) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    for example in raw_examples:
        print(example)

    raise ValueError(
        "ALQ_E raw formatting differs."
    )


print(
    "PASS: Raw CSV formatting exact."
)


# ============================================================
# 30. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0


for row in raw_rows:

    for value in row.values():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


print(
    "Unwanted '.0' values:",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unwanted .0 formatting remains."
    )


print(
    "PASS: No unwanted .0 values."
)


# ============================================================
# 31. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for row in raw_rows:

    for value in row.values():

        lower = value.lower()

        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 32. TINY ARTIFACT TEXT CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny artifact remains in CSV."
    )


print(
    "PASS: No tiny artifact remains."
)


# ============================================================
# 33. FINAL ZERO CHECK
# ============================================================

if int(
    csv_df["ALQ120Q"]
    .eq(0)
    .sum()
) != 1111:

    raise ValueError(
        "Final ALQ120Q zero count differs."
    )


if int(
    csv_df["ALQ140Q"]
    .eq(0)
    .sum()
) != 2036:

    raise ValueError(
        "Final ALQ140Q zero count differs."
    )


print(
    "PASS: Final restored-zero counts exact."
)


# ============================================================
# 34. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("ALQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nValidated tiny -> zero:"
)

print(
    "ALQ120Q = 1,111"
)

print(
    "ALQ140Q = 2,036"
)

print(
    "TOTAL   = 3,147"
)


print(
    "\nGenuine decimal variables:",
    0
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "ALQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 ALQ_E XPT -> CSV
Rows: 5,707
Columns: 9
PASS: ALQ_E structure = 5,707 rows x 9 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifacts: 3,147

Tiny values by variable:
ALQ140Q    2036
ALQ120Q    1111

PASS: Exactly 3,147 tiny artifacts found in ALQ120Q and ALQ140Q only.
PASS: Tiny values occur only in CDC zero-permitted variables.

--- ZERO RESTORATION ---
ALQ120Q   : 1,111 tiny -> 0
ALQ140Q   : 2,036 tiny -> 0
PASS: All 3,147 validated zeros restored.
PASS: ALQ120Q zeros = 1,111.
PASS: ALQ140Q zeros = 2,036.

Remaining tiny artifacts: 0
PASS: No tiny artifacts remain.

PASS: SEQN = 41475-51623, no missing, no duplicates.

Genuine fractional variables: {}
PASS: Genuine decimal variables = 0.

PASS: ALQ101 matches CDC.
PASS: ALQ110 matches CDC.

PASS: ALQ120Q
      0-365 = 4,423
      0 = 1,111
      999 = 11
      Missing = 1,273
PASS: ALQ120U matches CDC.
PASS: ALQ130 matches CDC.

PASS: ALQ140Q
      0-365 = 3,306
      0 = 2,036
      9999 = 1